In [2]:
import os
import pandas as pd
import plotly.express as px

# Notebooks start inside the notebooks/ folder, but our file paths start from the repo root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print(os.getcwd())

a:\AkshatKansal\projects\GLOF_risk-analysis-system\GLOF-RAS


In [3]:
labels = pd.read_csv("data/labels/dangerous_lake_labels.csv")
growth = pd.read_csv("outputs/tables/stage2_features_growth.csv")
gee = pd.read_csv("outputs/tables/stage2_features_terrain_rain.csv")

df = labels.merge(growth, on="lake_id").merge(gee, on="lake_id")
FEATURES = ["growth_rate", "acceleration", "slope_ring_mean_deg", "rain_max5d_mm", "rain_extreme_days"]

usable = df[df.n_valid_years >= 3].copy()
usable["group"] = usable.label.map({0: "unflagged", 1: "dangerous"})
print(df.shape, "->", len(usable), "usable lakes")
usable.head(20)

(190, 20) -> 173 usable lakes


,lake_id,name,lat,lon,basin,inventory_year,area,label,on_official_list,hazard_rank,...,n_valid_years,area_window_mean_km2,growth_rate,acceleration,slope_ring_mean_deg,steep_frac,rain_max5d_mm,rain_extreme_days,rain_p95_wetday_mm,group
0,I2015_GL086304E28374N,NaN,28.374033,86.304745,Koshi,2015,4.019206,1,Y,II,...,4,2.453912,1.073274e-04,-4.293400e-04,20.493165,0.268667,176.072206,2.50,49.213158,dangerous
1,I2015_GL085870E28360N,NaN,28.360000,85.870494,Koshi,2015,4.820760,1,Y,II,...,4,3.050653,1.069619e-07,-7.485042e-04,19.753164,0.274374,148.696411,3.25,50.645664,dangerous
2,I2015_GL085838E28322N,NaN,28.322233,85.838129,Koshi,2015,5.411391,1,Y,II,...,4,4.341411,1.754700e-04,-2.832421e-04,19.331667,0.257789,164.906822,2.75,57.480192,dangerous
4,I2015_GL086476E27861N,NaN,27.861020,86.476619,Koshi,2015,1.604792,1,Y,I,...,4,1.440570,-2.041321e-04,1.224418e-04,33.804252,0.653152,163.187570,3.25,63.660914,dangerous
5,I2015_GL086612E27779N,NaN,27.779128,86.612362,Koshi,2015,1.218551,1,Y,I,...,4,1.070100,9.900136e-04,1.779721e-07,25.629006,0.435489,153.218879,3.50,48.318554,dangerous
6,I2015_GL086925E27898N,NaN,27.898525,86.925096,Koshi,2015,1.342831,1,Y,I,...,4,1.056577,2.086482e-02,-5.374781e-02,24.794727,0.405322,189.543800,3.50,59.203660,dangerous
7,I2015_GL087632E27729N,NaN,27.729293,87.632834,Koshi,2015,0.026174,1,Y,III,...,4,0.024287,-2.545494e-02,4.872727e-01,21.247640,0.240778,164.245954,1.50,70.967572,dangerous
8,I2015_GL087596E27705N,NaN,27.705346,87.596579,Koshi,2015,0.031545,1,Y,I,...,4,0.014929,-2.061236e-02,-2.650009e-02,31.759129,0.582531,173.894338,2.00,69.915933,dangerous
10,I2015_GL086858E27687N,NaN,27.687513,86.858807,Koshi,2015,0.305487,1,Y,I,...,4,0.280164,-2.207483e-03,6.307118e-04,27.701492,0.432128,214.578882,4.25,72.946667,dangerous
11,I2015_GL086928E27850N,NaN,27.850246,86.928650,Koshi,2015,0.452416,1,Y,I,...,4,0.363113,-4.049472e-04,2.915469e-03,21.190553,0.273849,189.543800,3.50,59.203660,dangerous


In [3]:
usable[FEATURES].isna().sum()

growth_rate             0
acceleration           11
slope_ring_mean_deg     0
rain_max5d_mm           0
rain_extreme_days       0
dtype: int64

In [4]:
usable[FEATURES].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
growth_rate,173.0,-0.02,0.13,-0.50,-0.03,-0.0,0.01,0.50
acceleration,162.0,-0.03,0.20,-0.50,-0.06,-0.0,0.01,0.50
slope_ring_mean_deg,173.0,24.15,5.80,4.90,21.19,25.2,28.03,38.08
rain_max5d_mm,173.0,148.34,38.74,71.34,117.77,148.7,169.27,246.21
rain_extreme_days,173.0,3.04,0.87,1.25,2.25,3.0,3.75,5.25


In [5]:
usable.groupby("group")[FEATURES].median().round(3)

,growth_rate,acceleration,slope_ring_mean_deg,rain_max5d_mm,rain_extreme_days
group,,,,,
dangerous,-0.001,-0.001,25.95,164.254,3.25
unflagged,-0.007,-0.007,25.10,143.007,3.00


In [6]:
for f in FEATURES:
    fig = px.box(usable, x="group", y=f, color="group", points="all",
                 hover_name="lake_id", title=f)
    fig.show()

In [7]:
df[df.lake_id.str.startswith("SPECIAL")][["lake_id"] + FEATURES].round(3)

,lake_id,growth_rate,acceleration,slope_ring_mean_deg,rain_max5d_mm,rain_extreme_days
47,SPECIAL_SOUTH_LHONAK,0.001,-0.001,17.475,135.625,2.00
48,SPECIAL_THAME,0.012,-0.041,24.023,129.908,1.75
